In [ ]:
# SPIKE SORTING
#
# This notebook performs spike sorting and unit curation on the 
# preprocessed electrophysiological recordings generated by 01_preprocess_ephys.ipynb.
#
# Input structure:
# DATA/
# └── ephys/
#     └── <mouse_id>/
#         ├── spike/
#         │   ├── binary recording files
#         │   └── meta.json
#         ├── exploration_segments.csv
#         └── LED_info.csv
#
# The spike-band recordings were previously synchronized, quality-controlled,
# and filtered in 01_preprocess_ephys_P.ipynb. The corresponding meta.json
# stores the anatomical assignment of electrode channels to PFC and RSC.
#
# Workflow:
# 1. Load the preprocessed binary recording and channel assignments.
# 2. Select the PFC or RSC channels for spike sorting.
# 3. Whiten the recording and select the four highest-amplitude channels.
# 4. Assign probe geometry and run MountainSort5 spike sorting.
# 5. Compute waveform, template, and correlogram features for unit QC.
# 6. Identify and merge candidate duplicate units.
# 7. Apply quality-control criteria to retain well-isolated units.
# 8. Save the curated sorting and generate a SpikeInterface report.
# 9. Calculate firing rate and spike-width features and classify units
#    as fast-spiking (FS) or regular-spiking (RS).
#
# Main outputs:
# - Curated spike sorting
# - Spike-sorting quality-control report
# - <mouse_id>_<region>_units_types.csv

In [ ]:
from pathlib import Path
import json
import spikeinterface as si
import numpy as np
import sys

REPO_ROOT = Path("..").resolve()
sys.path.append(str(REPO_ROOT))
DATA_DIR = REPO_ROOT / "DATA"
EPHYS_DIR = DATA_DIR / "ephys"

MOUSE_ID = "<MOUSE_ID>"

RECORDING_DIR = EPHYS_DIR / MOUSE_ID
SPIKE_DIR = RECORDING_DIR / "spike"

spike_recording = si.load(SPIKE_DIR)

with (SPIKE_DIR / "meta.json").open("r") as file:
    spike_metadata = json.load(file)

# Recover anatomical channel assignments from preprocessing metadata
anatomical_assignment = spike_metadata["anatomical_assignment"]

pfc_channels = anatomical_assignment["pfc_channels"]
rsc_channels = anatomical_assignment["rsc_channels"]

print("Spike recording:")
print(spike_recording)

print("\nPFC channels:")
print(pfc_channels)

print("\nRSC channels:")
print(rsc_channels)

In [ ]:
# Select the regional recording used for spike sorting

pfc_recording = spike_recording.select_channels(pfc_channels)
rsc_recording = spike_recording.select_channels(rsc_channels)

selected_recording = rsc_recording #choose
region = "<REGION>"

print(f"\nSorting {region} units")
print(f"Number of channels: {selected_recording.get_num_channels()}")

In [ ]:
import spikeinterface.preprocessing as spre
from utils.ephys import plot_recording_with_amplitude

# Inspect the selected recording before whitening
plot_recording_with_amplitude(
    selected_recording,
    channel_idx=0,
    time_range=(0, 60),
)

# Whiten the recording before spike sorting
whitened_recording = spre.whiten(selected_recording)

# Inspect the recording after whitening
plot_recording_with_amplitude(
    whitened_recording,
    channel_idx=0,
    time_range=(0, 60),
    y_label=r"$\sigma$",
)

In [ ]:
from utils.ephys import select_top_channels

selected_recording_4ch, selected_channels = select_top_channels(
    whitened_recording,
    n_channels=4,
)

print("Selected channels:", selected_channels)

In [ ]:
# Output directory for the selected, whitened recording
output_recording_dir = RECORDING_DIR / f"{region}_selected_recording_4ch"

si.write_binary_recording(
    recording=selected_recording_4ch,
    file_paths=output_recording_dir,
    dtype="float32",
)

print(f"Saved selected recording to: {output_recording_dir}")

In [ ]:
from utils.ephys import add_probe_geometry

# Assign the electrode positions required for sorting

channel_positions = np.array([
    [0, 0],
    [20, 0],
    [0, 20],
    [20, 20],
])

selected_recording_4ch = add_probe_geometry(
    selected_recording_4ch,
    positions=channel_positions,
)

In [ ]:
import spikeinterface.sorters as ss

ss.get_default_sorter_params('mountainsort5')

In [ ]:
sorter_params = {
    "scheme": "2",
    "detect_threshold": 4.6,
    "detect_sign": -1,
    "detect_time_radius_msec": 1.5,
    "snippet_T1": 20,
    "snippet_T2": 20,
    "npca_per_channel": 8,
    "npca_per_subdivision": 15,
    "snippet_mask_radius": 20,
    "scheme1_detect_channel_radius": 50,
    "scheme2_phase1_detect_channel_radius": 50,
    "scheme2_detect_channel_radius": 30,
    "scheme2_max_num_snippets_per_training_batch": 1000,
    "whiten": False,
    "filter": False,
    "delete_temporary_recording": True,
    "pool_engine": "process",
    "n_jobs": 1,
    "chunk_duration": "1s",
    "progress_bar": True,
    "mp_context": None,
    "max_threads_per_worker": 1,
}

In [ ]:
# Run MountainSort5 on the selected four-channel recording

sorting = ss.run_sorter(
    sorter_name="mountainsort5",
    recording=selected_recording_4ch,
    remove_existing_folder=True,
    output_folder= RECORDING_DIR / f'{region}_mountainsort5_output',
    **sorter_params,
)

print(f"Detected units: {sorting.unit_ids}")

for unit_id in sorting.unit_ids:
    spike_train = sorting.get_unit_spike_train(unit_id)

    print(
        f"Unit {unit_id}: "
        f"{len(spike_train):,} spikes"
    )

In [ ]:
from spikeinterface import create_sorting_analyzer

# Compute waveform, firing, and template features for post-sorting unit QC

sorting_analyzer = create_sorting_analyzer(
    sorting,
    selected_recording_4ch,
    format="memory",
)

sorting_analyzer.compute(
    "random_spikes",
    method="uniform",
    max_spikes_per_unit=500,
)

sorting_analyzer.compute(
    "waveforms",
    ms_before=1.0,
    ms_after=2.0,
)

sorting_analyzer.compute(
    "correlograms",
    window_ms=100.0,
    bin_ms=1.0,
)

sorting_analyzer.compute("templates")
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("noise_levels")

import spikeinterface.widgets as sw

# Visualize unit templates and auto-/cross-correlograms for QC

sw.plot_unit_templates(
    sorting_analyzer,
    unit_ids=sorting.unit_ids,
)

sw.AutoCorrelogramsWidget(
    sorting_analyzer,
    unit_ids=sorting.unit_ids,
)

sw.CrossCorrelogramsWidget(
    sorting_analyzer,
)

In [ ]:
# Identify candidate unit groups that may represent the same neuron based on
# template similarity and correlogram features

from spikeinterface.curation import compute_merge_unit_groups

merge_groups = compute_merge_unit_groups(
    sorting_analyzer,
    preset="similarity_correlograms",
    resolve_graph=True,
)

if merge_groups:
    print("Merge groups:", merge_groups)
else:
    print("No merge groups identified.")

In [ ]:
# Merge units identified as likely duplicates and update the sorting

if merge_groups:
    sorting_analyzer = sorting_analyzer.merge_units(merge_groups)

sorting = sorting_analyzer.sorting

sorting_analyzer.compute(
    "random_spikes",
    method="uniform",
    max_spikes_per_unit=500,
)

# Recompute unit features after merging to ensure they reflect the updated sorting

sorting_analyzer.compute(
    "waveforms",
    ms_before=1.0,
    ms_after=2.0,
)

sorting_analyzer.compute(
    "correlograms",
    window_ms=100.0,
    bin_ms=1.0,
)

sorting_analyzer.compute("templates")
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("noise_levels")

sw.AutoCorrelogramsWidget(
    sorting_analyzer,
    unit_ids=sorting.unit_ids,
)

# Compute quality metrics used to assess the detected units

quality_metrics = sorting_analyzer.compute(
    "quality_metrics",
    metric_names=[
        "snr",
        "isi_violation",
        "presence_ratio",
        "amplitude_cutoff",
    ],
)

metrics_df = quality_metrics.get_data()

spike_counts = sorting.get_total_num_spikes()
recording_duration = selected_recording_4ch.get_total_duration()

metrics_df["num_spikes"] = [
    spike_counts[unit_id]
    for unit_id in metrics_df.index
]

metrics_df["firing_rate"] = (
    metrics_df["num_spikes"] / recording_duration
)

metrics_df["isi_violations_ratio"] = (
    metrics_df["isi_violations_count"]
    / metrics_df["num_spikes"]
)

metrics_df

In [ ]:
# Unit-quality thresholds
SNR_MIN = 3.0
ISI_VIOLATION_MAX = 0.01
PRESENCE_RATIO_MIN = 0.9
AMPLITUDE_CUTOFF_MAX = 0.2

good_units = metrics_df.query(
    "snr > @SNR_MIN & "
    "isi_violations_ratio < @ISI_VIOLATION_MAX & "
    "presence_ratio > @PRESENCE_RATIO_MIN & "
    "amplitude_cutoff < @AMPLITUDE_CUTOFF_MAX"
)

print(f"Accepted units: {len(good_units)}")

good_units

In [ ]:
# Keep only units that passed all quality-control criteria and save the curated sorting

good_unit_ids = good_units.index.tolist()

curated_sorting = sorting.select_units(good_unit_ids)

print("Curated units:", curated_sorting.unit_ids)

sorting_dir = RECORDING_DIR / f"{region}_curated_sorting" / "sorting"

curated_sorting = curated_sorting.save(folder=sorting_dir)

print(f"Curated sorting saved to: {sorting_dir}")

In [ ]:
# Export the QC-passed sorting to Phy2 for manual curation

# The sorting analyzer is created from the QC-passed units and the
# same recording used for spike sorting. Waveforms, templates,
# spike amplitudes, and principal components are precomputed because
# they are required/useful for Phy2 curation.
#
# The exported folder can then be opened in Phy2 for manual review,
# including manual unit splitting, merging, and rejection

phy_dir = RECORDING_DIR / f"{region}_phy"

phy_analyzer = si.create_sorting_analyzer(
    sorting=curated_sorting,
    recording=selected_recording_4ch,
    format="memory",
)

phy_analyzer.compute(
    "random_spikes",
    method="uniform",
    max_spikes_per_unit=500,
)

phy_analyzer.compute(
    "waveforms",
    ms_before=1.0,
    ms_after=2.0,
)

phy_analyzer.compute("templates")
phy_analyzer.compute("noise_levels")
phy_analyzer.compute("spike_amplitudes")

phy_analyzer.compute(
    "principal_components",
    n_components=5,
    mode="by_channel_local",
)

from spikeinterface.exporters import export_to_phy

export_to_phy(
    sorting_analyzer=phy_analyzer,
    output_folder=phy_dir,
)

print(f"Phy2 sorting exported to: {phy_dir}")
print("Perform manual curation in Phy2 before continuing.")

In [ ]:
# Re-import the manually curated sorting from Phy2

# Phy2 modifies the exported spike-sorting files during manual
# curation. Re-import the result into SpikeInterface so that all
# subsequent analyses use the manually curated units.

from spikeinterface.extractors import PhySortingExtractor

phy_dir = RECORDING_DIR / f"{region}_phy"
curated_sorting = PhySortingExtractor(phy_dir)

print("Manually curated units:", curated_sorting.unit_ids)

In [ ]:
# Compute post-curation features for the units retained after quality control

curated_analyzer = create_sorting_analyzer(
    curated_sorting,
    selected_recording_4ch,
    format="memory",
)

curated_analyzer.compute(
    "random_spikes",
    method="uniform",
    max_spikes_per_unit=500,
)

curated_analyzer.compute(
    "waveforms",
    ms_before=1.0,
    ms_after=2.0,
)

curated_analyzer.compute(
    "correlograms",
    window_ms=100.0,
    bin_ms=1.0,
)

curated_analyzer.compute("templates")
curated_analyzer.compute("template_similarity")
curated_analyzer.compute("noise_levels")

# Generate a report summarizing the curated spike-sorting results

from spikeinterface.exporters import export_report
import shutil

report_dir = RECORDING_DIR / f"{region}_report"

if report_dir.exists():
    shutil.rmtree(report_dir)
    
export_report(
    sorting_analyzer=curated_analyzer,
    output_folder=report_dir,
)

print(f"Sorting report saved to: {report_dir}")

In [ ]:
# Compute template metrics and derive firing rate and spike-width features for each unit

template_metrics = curated_analyzer.compute("template_metrics")
units_df = template_metrics.get_data().copy()

units_df.index.name = "unit_id"
units_df = units_df.reset_index()

recording_duration = selected_recording_4ch.get_total_duration()

units_df["firing_rate"] = [
    len(curated_sorting.get_unit_spike_train(unit_id))
    / recording_duration
    for unit_id in units_df["unit_id"]
]

units_df["spike_width_ms"] = (
    units_df["peak_to_valley"] * 1000
)

units_df["neuron_type"] = np.where(
    (units_df["spike_width_ms"] < 0.3)
    & (units_df["firing_rate"] > 10),
    "FS",
    "RS",
)
output_file = RECORDING_DIR / f"{MOUSE_ID}_{region}_units_types.csv"

units_df.to_csv(
    output_file,
    index=False,
)

print(f"Saved unit metrics to: {output_file}")